# Pathways Checks

Checks relationships and invariants around pathways, entrances, and platform connectivity.

In [1]:
from pathlib import Path
import sys

_current = Path.cwd().resolve()
for _candidate in [_current, *_current.parents]:
    if (_candidate / "data_validation" / "checks" / "commons.py").exists():
        _project_root = _candidate
        break
else:
    raise FileNotFoundError("data_validation/checks/commons.py not found.")

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from data_validation.checks.commons import (
    BASE,
    PATHWAYS_FILE,
    PW_PAIR,
    STOPS_FILE,
    TRANSFERS_FILE,
    build_graph_and_coverage,
    check_missing_files,
    iter_pathway_pairs,
    load_pathway_ids,
    load_platform_pairs_present,
    load_platforms_by_name,
    load_stop_ids,
    load_stop_names,
    load_stops_info,
    load_transfer_pairs,
    ordered_stop_ids,
    read_dict_rows,
)

In [7]:
def main() -> None:
    check_missing_files([PATHWAYS_FILE, STOPS_FILE, TRANSFERS_FILE])

    # Check that each (from_stop_id, to_stop_id) from transfers.txt has a
    # corresponding pathway_id = "PW.{from_stop_id}_{to_stop_id}" in pathways.txt
    pathways_ids = load_pathway_ids(PATHWAYS_FILE)
    transfers_pairs = list(load_transfer_pairs(TRANSFERS_FILE))
    expected_ids = [f"PW.{a}_{b}" for a, b in transfers_pairs]
    missing_transfers = [(a, b, eid) for (a, b), eid in zip(transfers_pairs, expected_ids) if eid not in pathways_ids]

    print(f"----- All transfers are within pathways.txt? -----")
    print(f"Total transfers: {len(transfers_pairs)}")
    print(f"Total pathways: {len(pathways_ids)}")

    if not missing_transfers:
        print("All correct: all rows in transfers have a corresponding pathway (format 'PW.{from_stop_id}_{to_stop_id}').")
    else:
        print(f"MISSING {len(missing_transfers)} pathways for specific transfers (showing all):")
        for a, b, eid in missing_transfers:
            print(f"- from_stop_id={a!r}, to_stop_id={b!r} -> expected pathway_id={eid!r}")

    stops_info_with_coords = load_stops_info(STOPS_FILE)
    candidate_pids = {pid for pid in pathways_ids if PW_PAIR.match(pid) is not None}
    missing_inverse = []
    for pid in sorted(candidate_pids):
        m = PW_PAIR.match(pid)
        a, b = m.group('a'), m.group('b')
        reverse_id = f"PW.{b}_{a}"
        if reverse_id not in pathways_ids:
            missing_inverse.append((pid, reverse_id, a, b))

    # Check symmetry in pathways: for each PW.x_y, check that PW.y_x exists.
    # If the inverse is missing, show stop_name and coordinates (lat/lon) for a and b.
    print(f"\n----- Each pathway 'PW.a_b' has its inverse 'PW.b_a'? -----")
    print(f"Candidates with format 'PW.a_b': {len(candidate_pids)}")
    if not missing_inverse:
        print("All correct: for each pathway 'PW.x_y' there also exists 'PW.y_x'.")
    else:
        print(f"MISSING {len(missing_inverse)} inverse pathways (showing all):")
        for pid, rid, a, b in missing_inverse:
            print(f"- Exists {pid!r} but missing its inverse {rid!r}")
            a_name, a_lat, a_lon = stops_info_with_coords.get(a, ("(no name)", "", ""))
            b_name, b_lat, b_lon = stops_info_with_coords.get(b, ("(no name)", "", ""))
            print(f"  · {a} — {a_name} (lat={a_lat}, lon={a_lon})")
            print(f"  · {b} — {b_name} (lat={b_lat}, lon={b_lon})")

    traversal_by_pid = {}
    for r in read_dict_rows(PATHWAYS_FILE):
        pid = r.get('pathway_id', '').strip()
        if pid in candidate_pids:
            traversal_by_pid[pid] = r.get('traversal_time', '').strip()

    compared_pairs = set()
    mismatches = []
    missing_reverse_pairs = 0
    for pid, a, b in iter_pathway_pairs(PATHWAYS_FILE):
        reverse_id = f"PW.{b}_{a}"
        pair_key = tuple(sorted((pid, reverse_id)))
        if pair_key in compared_pairs:
            continue
        compared_pairs.add(pair_key)
        if reverse_id not in traversal_by_pid:
            missing_reverse_pairs += 1
            continue
        t_ab = traversal_by_pid.get(pid, '')
        t_ba = traversal_by_pid[reverse_id]
        if t_ab != t_ba:
            mismatches.append((pid, t_ab, reverse_id, t_ba))

    print(f"\n----- traversal_time is the same for both directions? -----")
    print(f"Pairs compared (with reverse present): {len(compared_pairs) - missing_reverse_pairs}")
    if missing_reverse_pairs:
        print(f"Skipped {missing_reverse_pairs} pairs because reverse pathway is missing.")
    if not mismatches:
        print("All correct: traversal_time matches between 'PW.a_b' and 'PW.b_a' for all comparable pairs.")
    else:
        print(f"MISSING equal traversal_time in {len(mismatches)} pathway pairs:")
        for pid, t_ab, reverse_id, t_ba in sorted(mismatches):
            print(f"- {pid}: traversal_time={t_ab!r} | {reverse_id}: traversal_time={t_ba!r}")

    total_rows = 0
    valid_pw_rows = 0
    missing_traversal = []
    non_numeric = []
    not_multiple_15 = []
    for r in read_dict_rows(PATHWAYS_FILE):
        total_rows += 1
        pid = r.get('pathway_id', '').strip()
        if not pid or PW_PAIR.match(pid) is None:
            continue
        valid_pw_rows += 1
        t_raw = r.get('traversal_time', '').strip()
        if not t_raw:
            missing_traversal.append(pid)
            continue
        try:
            t_val = int(t_raw)
        except Exception:
            non_numeric.append((pid, t_raw))
            continue
        if t_val % 15 != 0:
            not_multiple_15.append((pid, t_val))

    print(f"\n----- traversal_time is present, numeric, and multiple of 15 for all PW pathways? -----")
    print(f"Total rows in pathways: {total_rows}")
    print(f"Rows with pathway_id format 'PW.a_b': {valid_pw_rows}")
    if not missing_traversal and not non_numeric and not not_multiple_15:
        print("All correct: traversal_time is present, numeric, and multiple of 15 for all PW pathways.")
    else:
        if missing_traversal:
            print(f"MISSING traversal_time in {len(missing_traversal)} pathways:")
            for pid in sorted(missing_traversal):
                print(f"- {pid}")
        if non_numeric:
            print(f"NON-NUMERIC traversal_time in {len(non_numeric)} pathways:")
            for pid, raw in sorted(non_numeric):
                print(f"- {pid}: traversal_time={raw!r}")
        if not_multiple_15:
            print(f"NOT multiple of 15 in {len(not_multiple_15)} pathways:")
            for pid, t_val in sorted(not_multiple_15):
                print(f"- {pid}: traversal_time={t_val}")

    # Check that each stop_id starting with 'E.' has at least one pathway with a platform '1.'
    # Requirement: pathway_id of the type 'PW.E.xxx_1.yyy' or 'PW.1.yyy_E.xxx'
    print(f"\n----- Each entrance is connected to a platform? -----")
    e_stops_info = {}
    for r in read_dict_rows(STOPS_FILE):
        sid = r.get('stop_id', '').strip()
        if not sid or not sid.startswith('E.') :
            continue
        e_stops_info[sid] = (r.get('stop_name', '').strip(), r.get('stop_lat', '').strip(), r.get('stop_lon', '').strip())

    covered_entrances = set()
    for pid in pathways_ids:
        m = PW_PAIR.match(pid)
        if not m:
            continue
        a, b = m.group('a'), m.group('b')
        if a.startswith('E.') and b.startswith('1.'):
            covered_entrances.add(a)
        elif b.startswith('E.') and a.startswith('1.'):
            covered_entrances.add(b)

    missing_entrances = [e for e in e_stops_info if e not in covered_entrances]
    print(f"Entrances (E.*): {len(e_stops_info)}")
    print(f"Entrances with at least one pathway to a platform (1.*): {len(covered_entrances)}")
    if not missing_entrances:
        print("All correct: each E.* has at least one pathway 'PW.E.xxx_1.yyy' or 'PW.1.yyy_E.xxx'.")
    else:
        print(f"MISSING {len(missing_entrances)} E.* without any pathway to 1.* (showing all):")
        for e in missing_entrances:
            name, lat, lon = e_stops_info.get(e, ("(no name)", "", ""))
            print(f"- {e} — {name} (lat={lat}, lon={lon})")

    # Check that each stop_id starting with '1.' (platform) has at least one
    # pathway with an entrance 'E.'. That is, look for 'PW.1.xxx_E.yyy' or
    # 'PW.E.yyy_1.xxx' for each '1.xxx'. If one is missing, show the stop_name
    # and suggest a neighboring platform (1.*) with its entrance (E.*) and names.
    print(f"\n----- Each platform is connected to an entrance? -----")
    stop_ids = load_stop_ids(STOPS_FILE)
    stop_names = load_stop_names(STOPS_FILE)
    one_stops = ordered_stop_ids(s for s in stop_ids if s.startswith('1.'))
    _, one_to_entries, covered_platforms = build_graph_and_coverage(pathways_ids)
    missing_platforms = [s for s in one_stops if s not in covered_platforms]
    print(f"Total stops: {len(stop_ids)}")
    print(f"Platforms (1.*): {len(one_stops)}")
    print(f"Platforms with at least one pathway to an entrance (E.*): {len(covered_platforms)}")
    if not missing_platforms:
        print("All correct: each 1.* has at least one pathway 'PW.1.xxx_E.yyy' or 'PW.E.yyy_1.xxx'.")
    else:
        print(f"MISSING {len(missing_platforms)} platforms without any pathway to any entrance (showing all):")
        for s in missing_platforms:
            nom = stop_names.get(s, "(no name)")
            print(f"- {s} — {nom}")
            neigh = sorted(set()) if not False else []
            neigh = []
            for pid in pathways_ids:
                m = PW_PAIR.match(pid)
                if not m:
                    continue
                a, b = m.group('a'), m.group('b')
                if a == s and b.startswith('1.'):
                    neigh.append(b)
                elif b == s and a.startswith('1.'):
                    neigh.append(a)
            sugg = [n for n in sorted(set(neigh)) if n in one_to_entries and one_to_entries[n]]
            if sugg:
                print("    Connected to platforms that do have at least one entrance:")
                for n in sugg:
                    n_nom = stop_names.get(n, "(no name)")
                    entries = sorted(one_to_entries[n])
                    e = entries[0]
                    e_nom = stop_names.get(e, "(no name)")
                    print(f"    · {n} — {n_nom} -> entrance {e} — {e_nom}")
            else:
                print("    (Not connected to any platform 1.* that has an entrance E.*)")

    # Check that all platforms (1.*) with the same stop_name have pathways between each pair. Example: if 1.339, 1.434 and 1.1136 exist with the same name, pathways are needed between each pair (we only check one direction per pair; reciprocity is already validated in a previous cell).
    print(f"\n----- There is a pathway between each pair of platforms of the same stop? -----")
    name_to_ones = load_platforms_by_name(STOPS_FILE)
    platform_pairs = load_platform_pairs_present(PATHWAYS_FILE)
    total_ones = sum(len(v) for v in name_to_ones.values())
    groups_multi = {name: ids for name, ids in name_to_ones.items() if len(ids) >= 2}
    print(f"Total platforms (1.*): {total_ones}")
    print(f"Stops with multiple platforms (same name): {len(groups_multi)}")
    missing_pairs = []
    for name, ids in sorted(groups_multi.items()):
        for u, v in [(a, b) for i, a in enumerate(sorted(ids)) for b in sorted(ids)[i + 1:]]:
            if (u, v) not in platform_pairs:
                missing_pairs.append((name, u, v))
    if not missing_pairs:
        print("All correct: for each stop with multiple platforms, there are pathways between all platform pairs.")
    else:
        print(f"MISSING pathways between {len(missing_pairs)} platform pairs within the same stop (showing all):")
        for name, u, v in missing_pairs:
            print(f"- {name}: missing pathway between {u} and {v}")

main()

----- All transfers are within pathways.txt? -----
Total transfers: 60
Total pathways: 1063
All correct: all rows in transfers have a corresponding pathway (format 'PW.{from_stop_id}_{to_stop_id}').

----- Each pathway 'PW.a_b' has its inverse 'PW.b_a'? -----
Candidates with format 'PW.a_b': 1063
MISSING 1 inverse pathways (showing all):
- Exists 'PW.1.120_E.12001' but missing its inverse 'PW.E.12001_1.120'
  · 1.120 — Plaça de Sants (lat=41.375353, lon=2.138154)
  · E.12001 — Alcolea (escala mecànica) (lat=41.375457, lon=2.137077)

----- traversal_time is the same for both directions? -----
Pairs compared (with reverse present): 531
Skipped 1 pairs because reverse pathway is missing.
All correct: traversal_time matches between 'PW.a_b' and 'PW.b_a' for all comparable pairs.

----- traversal_time is present, numeric, and multiple of 15 for all PW pathways? -----
Total rows in pathways: 1063
Rows with pathway_id format 'PW.a_b': 1063
All correct: traversal_time is present, numeric, and 